# Seq2Seq


In [ ]:
import numpy as np

class GRU(object):
    def __init__(self, input_dim, hidden_dim, output_dim, scale=0.01):
        super(GRU, self).__init__()
        self.input_dim, self.hidden_dim, self.output_dim, self.scale = (
            input_dim,
            hidden_dim,
            output_dim,
            scale,
        )

        normal = lambda m, n: np.random.randn(m, n) * scale
        three = lambda: (
            normal(input_dim, hidden_dim),
            normal(hidden_dim, hidden_dim),
            np.zeros((1, hidden_dim)),
        )

        Wxu, Whu, bu = three()  # Update gate parameter
        Wxr, Whr, br = three()  # Reset gate parameter
        Wxh, Whh, bh = three()  # Candidate hidden state parameter

        Wy = normal(hidden_dim, output_dim)
        by = np.zeros((1, output_dim))

        (
            self.Wxu,
            self.Whu,
            self.bu,
            self.Wxr,
            self.Whr,
            self.br,
            self.Wxh,
            self.Whh,
            self.bh,
            self.Wy,
            self.by,
        ) = (Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy, by)

        self.params = [Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy, by]
        # [dWxu, dWhu, dbu, dWxr, dWhr, dbr, dWxh, dWhh, dbh, dWy,dby]
        self.grads = [np.zeros_like(param) for param in self.params]

        self.H = None
        # params = [Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy,by]
        # return params

    def reset_state(self, batch_size):
        self.H = np.zeros((batch_size, self.hidden_dim))

    def forward_step(self, X):
        Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy, by = self.params
        H = self.H  # previous state
        X = Xs[t]
        U = sigmoid(np.dot(X, Wxu) + np.dot(H, Whu) + bu)
        R = sigmoid(np.dot(X, Wxr) + np.dot(H, Whr) + br)
        H_tilda = np.tanh(np.dot(X, Wxh) + np.dot(R * H, Whh) + bh)
        H = U * H + (1 - U) * H_tilda
        Y = np.dot(H, Wy) + by

        Hs[t] = H
        Ys.append(Y)
        Rs.append(R)
        Us.append(U)
        H_tildas.append(H_tilda)

    def forward(self, Xs):
        Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy, by = self.params
        if self.H is None:
            self.reset_state(Xs[0].shape[0])
        H = self.H
        Hs = {}
        Ys = []
        Hs[-1] = np.copy(H)
        Rs = []
        Us = []
        H_tildas = []

        for t in range(len(Xs)):
            X = Xs[t]
            U = sigmoid(np.dot(X, Wxu) + np.dot(H, Whu) + bu)
            R = sigmoid(np.dot(X, Wxr) + np.dot(H, Whr) + br)
            H_tilda = np.tanh(np.dot(X, Wxh) + np.dot(R * H, Whh) + bh)
            H = U * H + (1 - U) * H_tilda
            Y = np.dot(H, Wy) + by

            Hs[t] = H
            Ys.append(Y)
            Rs.append(R)
            Us.append(U)
            H_tildas.append(H_tilda)

        self.Ys, self.Hs, self.Rs, self.Us, self.H_tildas = Ys, Hs, Rs, Us, H_tildas
        return Ys, Hs
        # return Ys,Hs,(Rs,Us,H_tildas)

    def backward(self, dZs):  # Ys,loss_function):
        Wxu, Whu, bu, Wxr, Whr, br, Wxh, Whh, bh, Wy, by = self.params
        Ys, Hs, Rs, Us, H_tildas = self.Ys, self.Hs, self.Rs, self.Us, self.H_tildas

        dWxu, dWhu, dWxr, dWhr, dWxh, dWhh, dWy = (
            np.zeros_like(Wxu),
            np.zeros_like(Whu),
            np.zeros_like(Wxr),
            np.zeros_like(Whr),
            np.zeros_like(Wxh),
            np.zeros_like(Whh),
            np.zeros_like(Wy),
        )
        dbu, dbr, dbh, dby = (
            np.zeros_like(bu),
            np.zeros_like(br),
            np.zeros_like(bh),
            np.zeros_like(by),
        )

        dH_next = np.zeros_like(Hs[0])

        input_dim = Xs[0].shape[1]

        T = len(Xs)
        for t in reversed(range(T)):
            R = Rs[t]
            U = Us[t]
            H = Hs[t]
            X = Xs[t]
            H_tilda = H_tildas[t]
            H_pre = Hs[t - 1]

            dZ = dZs[t]
            # 输出f的模型参数的idu
            dWy += np.dot(H.T, dZ)
            dby += np.sum(dZ, axis=0, keepdims=True)

            # 隐状态h的梯度
            dH = np.dot(dZ, Wy.T) + dH_next

            #  H =  U H_pre+(1-U)H_tildas
            dH_tilda = dH * (1 - U)
            dH_pre = dH * U
            dU = H_pre * dH - H_tilda * dH

            # H_tilda = tanh(X Wxh+(R*H_)Whh+bh)
            dH_tildaZ = (1 - np.square(H_tilda)) * dH_tilda
            dWxh += np.dot(X.T, dH_tildaZ)
            dWhh += np.dot((R * H_pre).T, dH_tildaZ)
            dbh += np.sum(dH_tildaZ, axis=0, keepdims=True)

            dR = np.dot(dH_tildaZ, Whh.T) * H_pre
            dH_pre += np.dot(dH_tildaZ, Whh.T) * R

            # U = \sigma(UZ)   R = \sigma(RZ)
            dUZ = U * (1 - U) * dU
            dRZ = R * (1 - R) * dR

            dH_pre += np.dot(dUZ, Whu.T)
            dH_pre += np.dot(dRZ, Whr.T)

            # R = \sigma(X Wxr+H_ Whr + br)
            dWxr += np.dot(X.T, dRZ)
            dWhr += np.dot(H_pre.T, dRZ)
            dbr += np.sum(dRZ, axis=0, keepdims=True)

            dWxu += np.dot(X.T, dUZ)
            dWhu += np.dot(H_pre.T, dUZ)
            dbu += np.sum(dUZ, axis=0, keepdims=True)

            if True:
                dX_RZ = np.dot(dRZ, Wxr.T)
                dX_UZ = np.dot(dUZ, Wxu.T)
                dX_H_tildaZ = np.dot(dH_tildaZ, Wxh.T)
                dX = dX_RZ + dX_UZ + dX_H_tildaZ

            dH_next = dH_pre

        grads = [dWxu, dWhu, dbu, dWxr, dWhr, dbr, dWxh, dWhh, dbh, dWy, dby]
        for i, _ in enumerate(self.grads):
            self.grads[i] += grads[i]

        return self.grads
        # return [dWxu, dWhu, dbu, dWxr, dWhr, dbr, dWxh, dWhh, dbh, dWy,dby]

    def get_states(self):
        return self.Hs

    def get_outputs(self):
        return self.Ys

    def parameters(self):
        return self.params

## EncoderRNN

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import rnn as rnn


def one_hot(size, indices, expend=False):
    x = np.eye(size)[indices.reshape(-1)]
    if expend:
        x = np.expand_dims(x, axis=1)
    return x


class EncoderRNN(object):
    def __init__(self, input_size, hidden_size, num_layers=1):
        super(EncoderRNN, self).__init__()
        self.input_size, self.hidden_size = input_size, hidden_size
        self.num_layers = num_layers
        # self.embedding = Embedding(input_size, hidden_size)
        self.gru = GRU(input_size, hidden_size, 1)

    def word2vec(self, word_indices_input):
        return one_hot(self.input_size, word_indices_input, True)

    def forward(self, word_indices_input, hidden):
        # self.encode_input = one_hot(self.input_size,word_indices_input,True)
        self.encode_input = self.word2vec(word_indices_input)
        output, hidden = self.gru(self.encode_input, hidden)
        return output, hidden

    def __call__(self, word_indices_input, hidden):
        return self.forward(word_indices_input, hidden)

    def initHidden(self, batch_size=1):
        return np.zeros((self.num_layers, batch_size, self.hidden_size))

    def parameters(self):
        return self.gru.parameters()

    def backward(self, dhs):
        dinput, dhidden = self.gru.backward(dhs, self.encode_input)

ModuleNotFoundError: No module named 'modules'

## DecoderRNN

In [ ]:
class DecoderRNN(object):
    def __init__(
        self,
        input_size,
        hidden_size,
        output_size,
        num_layers=1,
        teacher_forcing_ratio=0.5,
    ):
        # super(DecoderRNN, self).__init__()
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.teacher_forcing_ratio = teacher_forcing_ratio

        self.gru = GRU(input_size, hidden_size, num_layers)
        self.out = Dense(hidden_size, output_size)

        self.layers = [self.gru, self.out]
        self._params = None

    def initHidden(self, batch_size=1):
        self.h_0 = np.zeros((self.num_layers, batch_size, self.hidden_size))

    def word2vec(self, input_t):
        return one_hot(self.input_size, input_t, True)

    def forward_step(self, input_t, hidden):
        gru_input = self.word2vec(input_t)
        self.input.append(gru_input)
        output_hs, hidden = self.gru(gru_input, hidden)
        output = self.out(output_hs[0])
        return output, hidden, output_hs[0]

    def forward(self, input_tensor, hidden):
        teacher_forcing_ratio = self.teacher_forcing_ratio
        use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False
        # use_teacher_forcing = True
        self.input = []

        output_hs = []
        output = []
        hidden_t = hidden
        h_0 = hidden.copy()

        input_t = np.array([SOS_token])
        # input_seq = []
        hs = []
        zs = []

        target_length = input_tensor.shape[0]
        for t in range(target_length):
            # input_seq.append(input_t[0])
            # print("type(input_t),input_t",type(input_t),input_t)

            output_t, hidden_t, output_hs_t = self.forward_step(input_t, hidden_t)

            # 保存每一时刻的计算结果
            hs.append(self.gru.hs)  # 隐状态
            zs.append(self.gru.zs)  # 中间变量
            output_hs.append(output_hs_t)
            output.append(output_t)

            if use_teacher_forcing:
                input_t = input_tensor[t]  # Teacher forcing
            else:
                input_t = np.argmax(output_t)  # 最大概率
                if input_t == EOS_token:
                    break
                input_t = np.array([input_t])

        output = np.array(output)
        self.output_hs = np.array(output_hs)
        self.h_0 = h_0
        self.hs = np.concatenate(hs, axis=1)
        self.zs = np.concatenate(zs, axis=1)

        # self.input_seq = input_seq
        # return  output,input_seq
        return output

    def __call__(self, input, hidden):
        return self.forward(input, hidden)

    def evaluate(self, hidden, max_length):
        # input:(1, batch_size=1, input_size)
        input = np.array([SOS_token])
        decoded_word_indices = []
        for t in range(max_length):
            output, hidden, _ = self.forward_step(input, hidden)
            output = np.argmax(output)
            if output == EOS_token:
                break
            else:
                decoded_word_indices.append(output)
                input = np.array([output])

        return decoded_word_indices
        # return  indexToSentence(output_lang,decoded_words)
        # return  indexToSentence(output_verb,decoded_words)

    def backward(self, dZs):
        dhs = []
        # output_hs,input_seq = self.output_hs,self.input_seq
        output_hs = self.output_hs
        input = np.concatenate(self.input, axis=0)

        for i in range(len(input)):
            self.out.x = output_hs[i]
            dh = self.out.backward(dZs[i])
            dhs.append(dh)
        dhs = np.array(dhs)

        self.gru.hs = self.hs
        self.gru.zs = self.zs
        self.gru.h = self.h_0

        dinput, dhidden = self.gru.backward(dhs, input)
        return dinput, dhidden

    #  def backward_dh(self,dZ):
    #      dh = self.out.backward(dZ)
    #      return dh

    def parameters(self):
        if self._params is None:
            self._params = []
            for layer in self.layers:
                for i, _ in enumerate(layer.params):
                    self._params.append([layer.params[i], layer.grads[i]])
        return self._params

## Train


In [ ]:
def train_step(
    input_tensor,
    target_tensor,
    encoder,
    decoder,
    encoder_optimizer,
    decoder_optimizer,
    loss_fn,
    reg,
    max_length=0,
):
    clip = 5.0
    #  encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.shape[0]  # input_tensor.size(0)

    loss = 0
    encode_input = input_tensor
    encoder_output, encoder_hidden = encoder(encode_input, None)
    decoder_hidden = encoder_hidden
    # output,input_seq = decoder(target_tensor, decoder_hidden)
    output = decoder(target_tensor, decoder_hidden)

    target = target_tensor.reshape(-1, 1)
    if output.shape[0] != target.shape[0]:
        target = target[: output.shape[0], :]
    loss, grad = loss_fn(output, target)
    loss /= output.shape[0]

    dinput, dhidden = decoder.backward(grad)
    encoder.backward(dhidden[0])  # ,encode_input)

    if reg is not None:
        loss += encoder_optimizer.regularization(reg)
        loss += decoder_optimizer.regularization(reg)

    util.clip_grad_norm_nn(encoder_optimizer.parameters(), clip, None)
    util.clip_grad_norm_nn(decoder_optimizer.parameters(), clip, None)

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss
    # return loss.item() / target_length

In [ ]:
import numpy as np
import time
import math
import matplotlib.pyplot as plt
%matplotlib inline

def timeSince(start):
    now = time.time()
    s = now - start  
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def trainIters(encoder, decoder, encoder_optimizer,decoder_optimizer,train_pairs,valid_pairs,print_every=1000, plot_every=100, reg =None):
    start = time.time()
    valid_losses = []
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every    
   
#    training_pairs = [tensorsFromPair(random.choice(train_pairs))      for i in range(n_iters)]
    training_pairs = train_pairs
 
    #criterion = nn.CrossEntropyLoss() #nn.NLLLoss()
    loss_fn =  util.rnn_loss_grad
    
    for iter in range(1, n_iters + 1):        
        pair = training_pairs[iter - 1]
        input_tensor,target_tensor = pair[0],pair[1]     
       
        loss = train_step(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, loss_fn,reg)
        if loss is None: continue
       
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
         #   plot_losses.append(loss)
            plot_loss_total = 0
            
            plt.plot(plot_losses)
            
            valid_losses.append(validation_loss(encoder, decoder, valid_pairs,20,reg))
            plt.plot(valid_losses)           
            plt.legend(["train_losses","valid_losses"])
            plt.show()

def validation_loss(encoder, decoder, valid_pairs,validation_size = None,reg =None):    
    if validation_size is not None:
        #valid_pairs = [tensorsFromPair(random.choice(valid_pairs))for i in range(validation_size)]
        valid_pairs = [random.choice(valid_pairs) for i in range(validation_size)]
    total_loss = 0
    loss_fn =  util.rnn_loss_grad
    
    teacher_forcing_ratio = decoder.teacher_forcing_ratio 
    decoder.teacher_forcing_ratio = 1.1
    for pair in valid_pairs:
        encode_input = pair[0]   
        target_tensor = pair[1]   
        
        encoder_output, encoder_hidden = encoder(encode_input, None)   
        decoder_hidden = encoder_hidden
        output = decoder(target_tensor, decoder_hidden) 

        
        target = target_tensor.reshape(-1,1)
        if output.shape[0]!= target.shape[0]:
            target = target[:output.shape[0],:]        
        loss,grad = loss_fn(output, target)    
        loss /=(output.shape[0])
       
        if reg is not None:
            params = encoder.parameters()+decoder.parameters()
            reg_loss =0
            for p,grad in params:            
                 reg_loss+= np.sum(p**2)
            loss += reg*reg_loss
    
        total_loss += loss
        
    decoder.teacher_forcing_ratio = teacher_forcing_ratio 
    return total_loss/len(valid_pairs)

## 字符翻譯

In [ ]:
SOS_token = 0
EOS_token = 1


class ChVerb:
    def __init__(self, name):
        self.name = name

        self.char2index = {"\t": 0, "\n": 1}
        self.index2char = {0: "\t", 1: "\n"}
        self.n_chars = 2  # Count SOS and EOS

    def addChars(self, chars):
        for char in chars:
            self.addChar(char)

    def addChar(self, char):
        if char not in self.char2index:
            self.char2index[char] = self.n_chars
            self.index2char[self.n_chars] = char
            self.n_chars += 1

In [ ]:
import numpy as np
import random
import re
import unicodedata

random.seed(1)

# Normalize every sentence
# you will normalize it to lower case,
# remove all non-character
# convert to ASCII from Unicode
# split the sentences, so you have each word in it.


def unicodeToAscii(sentence):
    return "".join(
        c
        for c in unicodedata.normalize("NFD", sentence)
        if unicodedata.category(c) != "Mn"
    )


def normalize_sentence(sentence):
    sentence = unicodeToAscii(sentence.lower().strip())
    # return sentence
    sentence = re.sub(r"([.!?])", r" \1", sentence)
    sentence = re.sub(r"[^a-zA-Z.!?]+", r" ", sentence)
    return sentence


MAX_LENGTH = 10

eng_prefixes = (
    "i am ",
    "i m ",
    "he is",
    "he s ",
    "she is",
    "she s ",
    "you are",
    "you re ",
    "we are",
    "we re ",
    "they are",
    "they re ",
)


def filterPair(p):
    return True
    return (
        len(p[0].split(" ")) < MAX_LENGTH
        and len(p[1].split(" ")) < MAX_LENGTH
        and p[1].startswith(eng_prefixes)
    )


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]


def prepareData(lang_from, lang_to, reverse=False):
    # construct verb
    if reverse:
        in_verb = ChVerb(lang_to)
        out_verb = ChVerb(lang_from)
    else:
        in_verb = ChVerb(lang_from)
        out_verb = ChVerb(lang_to)

    # read pairs
    data_path = "data/%s-%s/%s.txt" % (lang_to, lang_from, lang_to)
    with open(data_path, "r", encoding="utf-8") as f:
        lines = f.read().strip().split("\n")
        pairs = [[normalize_sentence(s) for s in l.split("\t")][:2] for l in lines]

        if reverse:
            pairs = [list(reversed(p)) for p in pairs]

    pairs = filterPairs(pairs)
    for pair in pairs:
        in_verb.addChars(pair[0])
        out_verb.addChars(pair[1])

    return in_verb, out_verb, pairs


in_verb, out_verb, pairs = prepareData("eng", "fra", True)  # False)

print("Read %s sentence pairs" % len(pairs))
print("Counted chars:")
print(in_verb.name, in_verb.n_chars)
print(out_verb.name, out_verb.n_chars)
for i in range(5):
    print(random.choice(pairs))
print(pairs[3])

Read 152583 sentence pairs
Counted chars:
fra 32
eng 32
['vous n etes pas ma mere .', 'you re not my mother .']
['ce que je veux pour noel c est que ma famille soit ici avec moi .', 'what i want for christmas is for my family to be here with me .']
['tout le monde est agace .', 'everybody is sore .']
['j ai vraiment besoin de votre aide .', 'i really do need your help .']
['il voulait reussir .', 'he wanted to succeed .']
['ca alors !', 'wow !']


In [ ]:
def indexToSentence(verb, indexes):
    sentense = [verb.index2char[idx] for idx in indexes]
    return "".join(sentense)


def indexesFromSentence(verb, sentence):
    return [verb.char2index[char] for char in sentence]


def tensorFromSentence(verb, sentence):
    indexes = indexesFromSentence(verb, sentence)
    indexes.append(EOS_token)
    return np.array(indexes).reshape(-1, 1)


#    return np.array(indexes,dtype = np.int64).reshape(-1,1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(in_verb, pair[0])
    target_tensor = tensorFromSentence(out_verb, pair[1])
    return (input_tensor, target_tensor)


print(pairs[3])
en_input, de_target = tensorsFromPair(pairs[3])  # random.choice(pairs))

print(en_input.shape)
print(de_target.shape)
print(en_input)
print(de_target)

['ca alors !', 'wow !']
(11, 1)
(6, 1)
[[ 6]
 [ 3]
 [ 4]
 [ 3]
 [13]
 [ 7]
 [ 9]
 [10]
 [ 4]
 [ 5]
 [ 1]]
[[10]
 [ 3]
 [10]
 [ 4]
 [ 9]
 [ 1]]


In [ ]:
from modules import train
from modules import layers
from modules import rnn
from modules import NeuralNetwork

# from rnn import *
# import util

hidden_size = 50  # 256
num_layers = 1

clip = 5.0  # 50.
learning_rate = 0.1
decoder_learning_ratio = 1.0
teacher_forcing_ratio = 0.5

encoder = EncoderRNN(in_verb.n_chars, hidden_size)
decoder = DecoderRNN(
    out_verb.n_chars, hidden_size, out_verb.n_chars, num_layers, teacher_forcing_ratio
)

momentum = 0.5
decay_every = 1000
encoder_optimizer = SGD(encoder.parameters(), learning_rate, momentum, decay_every)
decoder_optimizer = SGD(
    decoder.parameters(), learning_rate * decoder_learning_ratio, momentum, decay_every
)

reg = None  # 1e-2

if True:
    pairs = pairs[:80000]

np.random.shuffle(pairs)
train_n = (int)(len(pairs) * 0.98)
train_pairs = pairs[:train_n]
valid_pairs = pairs[train_n:]

n_iters = 50000
print_every, plot_every = 100, 100  # 10,10
idx_train_pairs = [tensorsFromPair(random.choice(train_pairs)) for i in range(n_iters)]
idx_valid_pairs = [tensorsFromPair(pair) for pair in valid_pairs]
trainIters(
    encoder,
    decoder,
    encoder_optimizer,
    decoder_optimizer,
    idx_train_pairs,
    idx_valid_pairs,
    print_every,
    plot_every,
    reg,
)

NameError: name 'Dense' is not defined